# Problem Set 2: Multivariate Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okuchap/GB656_2026_public/blob/main/problem-sets/03-02-template.ipynb)

In this problem set, you will predict hourly bike rentals using the Seoul Bike Sharing Demand dataset. Your goal is to fit and interpret a multivariate linear regression model that could help a bike-share operator forecast system-wide hourly demand.

## How to complete and submit this notebook

1. Open the template using the course Google Colab link.
2. Before editing, select **File > Save a copy in Drive**. Work only in the saved copy and rename it so the filename includes `PS2` and your name.
3. Run the cells from top to bottom. Complete every code and written-response **TODO** and replace each `...` in a code cell with your own code.
4. Do not edit or delete instructor-provided cells, including the two labeled result-check cells.
5. In Section 9, create one realistic scenario of your own and adapt the prediction workflow to it.
6. Before submitting, select **Runtime > Run all** and confirm that every requested output is visible and no cell reports an error. Save the notebook after the run finishes.
7. Click **Share**. Under **General access**, choose **Anyone with the link**, set the role to **Viewer**, and copy the sharing link.
8. Submit the link to your completed Drive copy as the **Website URL** in Canvas. Do not submit the original GitHub template link.
9. After the deadline, do not edit the submitted notebook unless the instructor asks you to resubmit.

Setup, data-loading, and selected display code are provided. You will complete the main preprocessing, plotting, model-fitting, result-extraction, and prediction steps, then interpret the results in your own words.

## Data

The instructor-provided helper below first looks for `data/SeoulBikeData.csv` in a local course repository. If no local copy is available—as in a fresh Colab runtime—it loads the same file from the public course repository on GitHub. You do not need to upload the dataset or mount Google Drive.

## 1. Import packages

Run this cell first.

In [ ]:
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.style.use("seaborn-v0_8-whitegrid")

## 2. Load the data

The original CSV uses a non-UTF-8 encoding, so the loading code uses `encoding="latin1"`. The helper accepts either the local path or the public URL as the input to `pd.read_csv()`.

In [ ]:
PUBLIC_REPOSITORY = "okuchap/GB656_2026_public"
PUBLIC_REVISION = "main"


def course_data_source(file_name):
    """Return a local course-data path when available, otherwise its public URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / file_name
        if local_path.is_file():
            return local_path

    encoded_name = quote(file_name)
    return (
        "https://raw.githubusercontent.com/"
        f"{PUBLIC_REPOSITORY}/{PUBLIC_REVISION}/data/{encoded_name}"
    )


data_source = course_data_source("SeoulBikeData.csv")
bikes_raw = pd.read_csv(data_source, encoding="latin1")
source_location = (
    "local course repository" if isinstance(data_source, Path) else "public GitHub repository"
)
print(
    f"Loaded {bikes_raw.shape[0]:,} rows and {bikes_raw.shape[1]} columns "
    f"from the {source_location}."
)
bikes_raw.head()

In [ ]:
bikes_raw.columns

## 3. Rename the columns

The original column names contain spaces, parentheses, and units. The provided code renames them to simple lowercase names.

In [ ]:
bikes = bikes_raw.copy()

bikes.columns = [
    "date",
    "bike_count",
    "hour",
    "temperature_c",
    "humidity_percent",
    "wind_speed_ms",
    "visibility_10m",
    "dew_point_c",
    "solar_radiation",
    "rainfall_mm",
    "snowfall_cm",
    "season",
    "holiday",
    "functioning_day",
]

bikes.head()

## 4. Preprocess the data

### Step 4A — Parse the dates and filter the rows

Complete the two marked lines to:

1. convert `date` with `pd.to_datetime(..., dayfirst=True)`; and
2. keep only rows for which `functioning_day` is `"Yes"`. Use `.copy()` after filtering.

In [ ]:
bikes["date"] = ...  # TODO: convert the date column; the day appears first

bikes = ...  # TODO: keep operating hours and make a copy

bikes.shape

### Step 4B — Create a holiday indicator

The `holiday` column contains text labels, but the regression needs a numeric feature. First, use `.value_counts()` to inspect the labels and their frequencies. Then create `is_holiday`, equal to 1 for `"Holiday"` and 0 for `"No Holiday"`.

The comparison creates `True` and `False` values; use `.astype(int)` to convert them to 1 and 0.

In [ ]:
holiday_counts = ...  # TODO: count the values in the holiday column
holiday_counts

In [ ]:
bikes["is_holiday"] = ...  # TODO: create the 0/1 indicator

bikes[["holiday", "is_holiday"]].head()

### Step 4C — Create season dummy variables

First inspect the season counts. Then encode the four seasons with `Winter` as the omitted baseline category.

The provided `season_order` puts `Winter` first. Complete the marked arguments in `pd.Categorical()` and `pd.get_dummies()`. With an intercept and four categories, the model should use exactly three season dummy variables.

In [ ]:
season_counts = ...  # TODO: count the values in the season column
season_counts

In [ ]:
season_order = ["Winter", "Spring", "Summer", "Autumn"]

bikes["season"] = pd.Categorical(
    ...,  # TODO: select the season column
    categories=...,  # TODO: use the supplied category order
)

season_dummies = pd.get_dummies(
    ...,  # TODO: encode the ordered season column
    prefix="season",
    drop_first=...,  # TODO: omit the Winter baseline
    dtype=int,
)

season_dummies.head()

### Step 4D — Summarize the working data

Use `.describe()` and round the result to two decimal places for the supplied columns.

In [ ]:
summary_columns = [
    "bike_count",
    "hour",
    "temperature_c",
    "humidity_percent",
    "wind_speed_ms",
    "rainfall_mm",
    "snowfall_cm",
    "is_holiday",
]

summary_stats = ...  # TODO: select, describe, and round to two decimals
summary_stats

### Question 4.1 — working dataset

**TODO:** In one or two sentences, report the number of observations after filtering, the number of holiday categories, and the average hourly bike count.

**Your answer:**  

## 5. Plot the data

### Step 5A — Bike rentals and temperature

Complete the scatterplot. Put `temperature_c` on the horizontal axis and `bike_count` on the vertical axis. Add clear axis labels and a title.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    ...,  # TODO: horizontal-axis data
    ...,  # TODO: vertical-axis data
    alpha=0.25,
)
ax.set_xlabel("Temperature (degrees C)")
ax.set_ylabel("Hourly rented bikes")
ax.set_title("Bike rentals vs. temperature")

plt.show()

### Step 5B — Average rentals by season

Complete the grouped summary and bar chart. The `season_means` Series should contain mean `bike_count` by `season`, sorted from the smallest mean to the largest.

In [ ]:
season_means = (
    bikes.groupby(..., observed=False)[...]  # TODO: group and select
    .mean()
    .sort_values()
)

season_means.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

season_means.plot(kind=..., ax=ax)  # TODO: make a bar chart
ax.set_xlabel("Season")
ax.set_ylabel("Average hourly rented bikes")
ax.set_title("Average bike rentals by season")
plt.xticks(rotation=0)

plt.show()

### Question 5.1 — exploratory patterns

**TODO:** Describe both plots. What relationships do you see between bike rentals, temperature, and season? Does either plot establish a causal relationship? Explain briefly.

**Your answer:**  

## 6. Fit the OLS regression

### Step 6A — Build the outcome and feature matrix

Use `bike_count` as the outcome. The supplied list fixes the numeric weather/time features for everyone. Complete the code to:

1. create `y`;
2. combine the numeric features and `season_dummies` into `X`; and
3. add an intercept column to `X` with `sm.add_constant()`.

In [ ]:
numeric_features = [
    "hour",
    "temperature_c",
    "humidity_percent",
    "wind_speed_ms",
    "rainfall_mm",
    "snowfall_cm",
    "is_holiday",
]

y = ...  # TODO: select bike_count as a Series
X = ...  # TODO: concatenate numeric features and season dummies column-wise
X = ...  # TODO: add the intercept column

X.head()

### Step 6B — Fit the model

Use `sm.OLS()` with the outcome first and feature matrix second, then call `.fit()`.

In [ ]:
ols_model = ...  # TODO: specify and fit the OLS model

print(ols_model.summary())

### Question 7.1 — intercept

**TODO:** Report the estimated intercept. What combination of feature values and baseline categories does it describe? Is that combination practically meaningful?

**Your answer:**  

### Question 7.2 — temperature

**TODO:** Report and interpret the coefficient on `temperature_c`, including its sign, size, and units. Use the phrase **holding the other included features fixed**.

**Your answer:**  

### Question 7.3 — season

**TODO:** Report and interpret the season coefficient you selected. Identify the omitted comparison category.

**Your answer:**  

### Question 7.4 — another weather feature

**TODO:** Report and interpret the non-temperature weather coefficient you selected, again holding the other included features fixed.

**Your answer:**  

### Question 7.5 — standard error

**TODO:** Report the standard error for the temperature coefficient. What does a standard error measure?

**Your answer:**  

### Question 7.6 — model fit

**TODO:** Report $R^2$ and interpret it in context. Explain why this in-sample value alone does not establish out-of-sample forecasting accuracy.

**Your answer:**  

## 8. Make the required predictions

The four required future scenarios are provided below. Retain these values so everyone analyzes the same cases.

In [ ]:
new_scenarios = pd.DataFrame(
    {
        "scenario": [
            "Spring morning commute",
            "Autumn evening commute",
            "Summer rainy afternoon",
            "Winter holiday evening",
        ],
        "hour": [8, 17, 13, 20],
        "temperature_c": [10, 22, 30, -3],
        "humidity_percent": [55, 60, 70, 45],
        "wind_speed_ms": [1.5, 2.0, 1.0, 2.5],
        "rainfall_mm": [0, 0, 2, 0],
        "snowfall_cm": [0, 0, 0, 1],
        "is_holiday": [0, 0, 0, 1],
        "season": ["Spring", "Autumn", "Summer", "Winter"],
    }
)

new_scenarios

### Step 8A — Encode and align the new features

The prediction matrix must use the same encoding, column names, and column order as the training matrix `X`. Complete these steps:

1. apply the full `season_order` to the new season column;
2. create the same three season dummies;
3. combine the numeric and dummy features;
4. add an intercept with `has_constant="add"`; and
5. reindex to `X.columns`, filling any missing columns with 0.

In [ ]:
new_scenarios["season"] = pd.Categorical(
    ...,  # TODO: select the new season column
    categories=...,  # TODO: reuse the full training category order
)

new_season_dummies = pd.get_dummies(
    ...,  # TODO: encode the new ordered season column
    prefix="season",
    drop_first=...,  # TODO: use the same baseline rule
    dtype=int,
)

new_X = ...  # TODO: concatenate numeric features and new season dummies
new_X = sm.add_constant(..., has_constant="add")  # TODO: add the intercept
new_X = ...  # TODO: align to X.columns and fill missing columns with 0

new_X

### Result check — do not edit

The next prediction call requires the new feature columns to match the fitted model columns in exactly the same order.

In [ ]:
assert new_X.columns.tolist() == X.columns.tolist(), (
    "Before predicting, align new_X to X.columns in the same order."
)
print("Required-scenario columns match the fitted model.")

### Step 8B — Generate the point predictions

Call the fitted model's `.predict()` method with `new_X` and store the results in `predicted_bike_count`.

In [ ]:
new_scenarios["predicted_bike_count"] = ...  # TODO: predict with new_X

new_scenarios[["scenario", "predicted_bike_count"]].round(2)

### Question 8.1 — required scenarios

**TODO:** Report the highest and lowest required predictions, including both the scenario names and predicted bike counts. Does this ranking make business sense? Explain briefly.

**Your answer:**  

## 9. Make one additional prediction

### Step 9A — Define your scenario

Create a one-row DataFrame named `my_scenario` for one realistic future situation of your choice. Follow the structure of `new_scenarios` and include these columns in the same order:

- `scenario`
- every column in `numeric_features`
- `season`

Replace every `...` below with a one-element list. Use an hour from 0 through 23, a temperature from -20 through 40 degrees C, humidity from 0 through 100, nonnegative wind/rain/snow values, `is_holiday` equal to 0 or 1, and one season from `season_order`.

In [ ]:
my_scenario = pd.DataFrame(
    {
        "scenario": ...,  # TODO: descriptive name in a one-element list
        "hour": ...,  # TODO
        "temperature_c": ...,  # TODO
        "humidity_percent": ...,  # TODO
        "wind_speed_ms": ...,  # TODO
        "rainfall_mm": ...,  # TODO
        "snowfall_cm": ...,  # TODO
        "is_holiday": ...,  # TODO
        "season": ...,  # TODO
    }
)

my_scenario

### Step 9B — Encode and align your scenario

The first line applies the full category order. Then adapt your Section 8 code to:

1. create `my_season_dummies`;
2. create `my_X` by combining the numeric and dummy features;
3. add the intercept with `has_constant="add"`; and
4. reindex to `X.columns` with missing values filled by 0.

Write the remaining code yourself by adapting the completed workflow above.

In [ ]:
my_scenario["season"] = pd.Categorical(
    my_scenario["season"],
    categories=season_order,
)

# TODO: Write the remaining encoding and alignment code here.

my_X

### Result check — do not edit

The next prediction call requires your feature columns to match the fitted model columns in exactly the same order.

In [ ]:
assert my_X.columns.tolist() == X.columns.tolist(), (
    "Before predicting, align my_X to X.columns in the same order."
)
print("Additional-scenario columns match the fitted model.")

### Step 9C — Generate your prediction

Call the fitted model's `.predict()` method with `my_X`, store the result in `predicted_bike_count`, and display the scenario name and prediction.

In [ ]:
my_scenario["predicted_bike_count"] = ...  # TODO: predict with my_X

my_scenario[["scenario", "predicted_bike_count"]].round(2)

### Question 9.1 — additional scenario

**TODO:** Describe your scenario, report its predicted bike count, and explain whether the prediction seems plausible.

**Your answer:**  

## 10. Business summary

### Question 10.1 — recommendation

**TODO:** Write a concise 4–6 sentence business summary that answers all of the following:

1. What did the model find, and which included features appear useful for predicting bike demand?
2. How could a bike-share operator use system-wide hourly demand predictions?
3. Would you use this model as a final operational forecasting tool? Why or why not?
4. What additional data or model features could improve it? Consider the linear treatment of hour and the absence of station-level information.

**Your answer:**  